Objectif : faire une classification à partir des images brutes + segmentation vaisseaux sanguins à partir de notre modèle pour déduire la maladie du patient.

Rappel : 
 - N : normal
 - A : age-related macular degeneration
 - G : glaucoma
 - D : diabetic retinopathy

In [1]:
import os
import sys
import matplotlib.pyplot as plt
import yaml

sys.path.append('..')

In [2]:
# Construction d'un simple CNN qui construit une représentation d'une image d'entrée quelconque

import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.norm1 = self._make_gn(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.norm2 = self._make_gn(out_channels)
        self.act = nn.SiLU(inplace=True)
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                self._make_gn(out_channels),
            )
        else:
            self.skip = nn.Identity()

    @staticmethod
    def _make_gn(channels):
        for g in (32, 16, 8, 4, 2, 1):
            if channels % g == 0:
                return nn.GroupNorm(g, channels)
        return nn.GroupNorm(1, channels)

    def forward(self, x):
        identity = self.skip(x)
        x = self.act(self.norm1(self.conv1(x)))
        x = self.drop(x)
        x = self.norm2(self.conv2(x))
        x = self.act(x + identity)
        return x


class SimpleImageCNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=4):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2, bias=False),
            ResidualBlock._make_gn(32),
            nn.SiLU(inplace=True),
        )

        self.layer1 = ResidualBlock(32, 64, stride=2, dropout=0.05)
        self.layer2 = ResidualBlock(64, 128, stride=2, dropout=0.10)
        self.layer3 = ResidualBlock(128, 256, stride=2, dropout=0.15)

        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x

## Chargement des données

In [3]:
# Chargement des chemins depuis la config
train_img_path = "../data/train/original/"
train_label_path = "../data/train/ground_truth/"
test_img_path = "../data/test/original/"
test_label_path = "../data/test/ground_truth/"

train_img_files = os.listdir(train_img_path)
train_label_files = os.listdir(train_label_path)
test_img_files = os.listdir(test_img_path)
test_label_files = os.listdir(test_label_path)

train_img = [os.path.join(train_img_path, file) for file in train_img_files]
train_label = [os.path.join(train_label_path, file) for file in train_label_files]
test_img = [os.path.join(test_img_path, file) for file in test_img_files]
test_label = [os.path.join(test_label_path, file) for file in test_label_files]

print(f'Number of training images: {len(train_img)}')
print(f'Number of training labels: {len(train_label)}')
print(f'Number of testing images: {len(test_img)}')
print(f'Number of testing labels: {len(test_label)}')

Number of training images: 600
Number of training labels: 600
Number of testing images: 200
Number of testing labels: 200


In [4]:
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode
from src.dataset.dataset import VesselDataset
from torch.utils.data import DataLoader, random_split

# Paramètres depuis la config
batch_size = 1
image_size = 512
train_split = 0.8

image_transform = transforms.Compose([
    transforms.Resize((image_size, image_size), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
])

label_transform = transforms.Compose([
    transforms.Resize((image_size, image_size), interpolation=InterpolationMode.NEAREST),
    transforms.ToTensor(),
])

# Dataset train/val (à partir du dossier train)
dataset = VesselDataset(
    train_img,
    train_label,
    image_transform=image_transform,
    label_transform=label_transform,
 )

train_size = int(train_split * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Dataset test (dossier test)
test_dataset = VesselDataset(
    test_img,
    test_label,
    image_transform=image_transform,
    label_transform=label_transform,
 )
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Dataset créé avec image_size={image_size}, batch_size={batch_size}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Dataset créé avec image_size=512, batch_size=1
Train: 480, Val: 120, Test: 200


In [5]:
import os
import time
import copy
import torch.optim as optim
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Mapping des classes
label_to_idx = {'N': 0, 'A': 1, 'G': 2, 'D': 3}
idx_to_label = {v: k for k, v in label_to_idx.items()}

def get_safe_device(preferred='cuda'):
    """Choisit un device utilisable. En cas d'erreur CUDA, bascule automatiquement sur CPU."""
    if preferred == 'cuda' and torch.cuda.is_available():
        try:
            _ = torch.zeros(1, device='cuda')
            return torch.device('cuda')
        except Exception as e:
            print(f"CUDA indisponible/instable ({e}). Bascule sur CPU.")
            return torch.device('cpu')
    return torch.device('cpu')

def build_model_input(images, vessel_masks, expected_in_channels):
    """Prépare l'entrée du modèle selon le nombre de canaux attendu.
    - 1 canal: masque vaisseaux seul
    - 3 canaux: image RGB seule
    - 4 canaux: image RGB + masque vaisseaux (1 canal)
    """
    if expected_in_channels == 1:
        return vessel_masks
    if expected_in_channels == 3:
        return images
    if expected_in_channels == 4:
        return torch.cat([images, vessel_masks], dim=1)
    raise ValueError(f"in_channels={expected_in_channels} non supporté (attendu: 1, 3 ou 4).")

def encode_labels(disease_letters, label_map):
    cleaned = [str(d).strip().upper() for d in disease_letters]
    unknown = [d for d in cleaned if d not in label_map]
    if unknown:
        raise ValueError(f"Labels inconnus trouvés: {unknown}")
    return torch.tensor([label_map[d] for d in cleaned], dtype=torch.long)

def validate_targets(targets, num_classes):
    tmin = int(targets.min().item())
    tmax = int(targets.max().item())
    if tmin < 0 or tmax >= num_classes:
        raise ValueError(
            f"Targets hors plage pour CrossEntropyLoss: min={tmin}, max={tmax}, "
            f"num_classes={num_classes}"
        )

def load_classification_checkpoint(checkpoint_path, model, optimizer=None, device='cpu'):
    """Charge un checkpoint de classification pour reprendre l'entraînement."""
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint introuvable: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    history = checkpoint.get('history', {})
    history = {
        'train_loss': history.get('train_loss', []),
        'train_acc': history.get('train_acc', []),
        'val_loss': history.get('val_loss', []),
        'val_acc': history.get('val_acc', []),
    }
    start_epoch = int(checkpoint.get('epoch', len(history['train_loss'])))

    print(f"Checkpoint chargé depuis: {checkpoint_path}")
    print(f"Epoch reprise: {start_epoch}")
    return model, optimizer, history, start_epoch

def train_classifier(
    model,
    train_loader,
    val_loader,
    n_epochs=10,
    learning_rate=1e-3,
    device=None,
    save_root_dir='../saved_models',
    expected_in_channels=3,
    num_classes=None,
    model_dir=None,
    optimizer=None,
    initial_history=None,
    start_epoch=0,
    ):
    """Entraîne un classifieur multi-classes, sauvegarde best model + checkpoint, et retourne modèle + historique."""
    if num_classes is None:
        if hasattr(model, 'classifier') and hasattr(model.classifier, 'out_features'):
            num_classes = model.classifier.out_features
        else:
            raise ValueError("Impossible d'inférer num_classes. Passe num_classes explicitement.")

    if device is None:
        device = get_safe_device('cuda')

    try:
        model = model.to(device)
    except Exception as e:
        print(f"Échec sur {device} ({e}). Retry sur CPU.")
        device = torch.device('cpu')
        model = model.to(device)

    if model_dir is None:
        run_name = f"classification_train_{time.strftime('%Y%m%d-%H%M%S')}"
        model_dir = os.path.join(save_root_dir, run_name)
    os.makedirs(model_dir, exist_ok=True)
    best_model_path = os.path.join(model_dir, 'best_model.pth')
    checkpoint_path = os.path.join(model_dir, 'checkpoint.pth')

    if optimizer is None:
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    if initial_history is None:
        history = {
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': [],
        }
    else:
        history = {
            'train_loss': list(initial_history.get('train_loss', [])),
            'train_acc': list(initial_history.get('train_acc', [])),
            'val_loss': list(initial_history.get('val_loss', [])),
            'val_acc': list(initial_history.get('val_acc', [])),
        }

    if start_epoch == 0 and len(history['train_loss']) > 0:
        start_epoch = len(history['train_loss'])

    best_val_acc = max(history['val_acc']) if len(history['val_acc']) > 0 else 0.0
    best_epoch = (history['val_acc'].index(best_val_acc) + 1) if len(history['val_acc']) > 0 else 0
    best_state = copy.deepcopy(model.state_dict())

    if start_epoch >= n_epochs:
        print(f"Aucun entraînement supplémentaire: start_epoch={start_epoch} >= n_epochs={n_epochs}")
        return model, history, model_dir

    criterion = nn.CrossEntropyLoss()

    for epoch in range(start_epoch + 1, n_epochs + 1):
        # --- Training ---
        model.train()
        running_loss, running_correct, running_total = 0.0, 0, 0

        train_iter = tqdm(
            train_loader,
            desc=f"Epoch {epoch:02d}/{n_epochs} [train]",
            leave=False,
            dynamic_ncols=True
        )
        for images, vessel_masks, disease_letters in train_iter:
            targets = encode_labels(disease_letters, label_to_idx)
            validate_targets(targets, num_classes)

            images = images.to(device)
            vessel_masks = vessel_masks.to(device)
            targets = targets.to(device)

            inputs = build_model_input(images, vessel_masks, expected_in_channels)

            optimizer.zero_grad()
            logits = model(inputs)
            if logits.shape[1] != num_classes:
                raise ValueError(
                    f"Sortie modèle incohérente: logits.shape[1]={logits.shape[1]} != num_classes={num_classes}"
                )

            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()

            preds = torch.argmax(logits, dim=1)
            bs = targets.size(0)
            running_loss += loss.item() * bs
            running_correct += (preds == targets).sum().item()
            running_total += bs

            train_iter.set_postfix(
                loss=f"{(running_loss / max(running_total, 1)):.4f}",
                acc=f"{(running_correct / max(running_total, 1)):.4f}"
            )

        train_loss = running_loss / max(running_total, 1)
        train_acc = running_correct / max(running_total, 1)

        # --- Validation ---
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0

        val_iter = tqdm(
            val_loader,
            desc=f"Epoch {epoch:02d}/{n_epochs} [val]",
            leave=False,
            dynamic_ncols=True
        )
        with torch.no_grad():
            for images, vessel_masks, disease_letters in val_iter:
                targets = encode_labels(disease_letters, label_to_idx)
                validate_targets(targets, num_classes)

                images = images.to(device)
                vessel_masks = vessel_masks.to(device)
                targets = targets.to(device)

                inputs = build_model_input(images, vessel_masks, expected_in_channels)
                logits = model(inputs)
                if logits.shape[1] != num_classes:
                    raise ValueError(
                        f"Sortie modèle incohérente: logits.shape[1]={logits.shape[1]} != num_classes={num_classes}"
                    )

                loss = criterion(logits, targets)

                preds = torch.argmax(logits, dim=1)
                bs = targets.size(0)
                val_loss_sum += loss.item() * bs
                val_correct += (preds == targets).sum().item()
                val_total += bs

                val_iter.set_postfix(
                    loss=f"{(val_loss_sum / max(val_total, 1)):.4f}",
                    acc=f"{(val_correct / max(val_total, 1)):.4f}"
                )

        val_loss = val_loss_sum / max(val_total, 1)
        val_acc = val_correct / max(val_total, 1)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Sauvegarde du meilleur modèle
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, best_model_path)

        # Sauvegarde du checkpoint complet à chaque epoch
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history,
            'best_val_acc': best_val_acc,
            'best_epoch': best_epoch,
            'num_classes': num_classes,
            'expected_in_channels': expected_in_channels,
            'label_to_idx': label_to_idx,
        }
        torch.save(checkpoint, checkpoint_path)

        print(
            f"Epoch {epoch:02d}/{n_epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    print(f"Meilleure val_acc: {best_val_acc:.4f} (epoch {best_epoch})")
    print(f"Best model: {best_model_path}")
    print(f"Checkpoint: {checkpoint_path}")
    return model, history, model_dir

def predict_and_evaluate(
    model,
    data_loader,
    expected_in_channels=3,
    device=None,
    label_map=None,
    ):
    """Prédit sur un DataLoader et retourne des métriques globales et par classe."""
    if label_map is None:
        label_map = label_to_idx
    inv_map = {v: k for k, v in label_map.items()}

    if device is None:
        device = get_safe_device('cuda')

    model = model.to(device)
    model.eval()

    all_targets = []
    all_preds = []

    with torch.no_grad():
        test_iter = tqdm(data_loader, desc='Test [predict]', leave=False, dynamic_ncols=True)
        for images, vessel_masks, disease_letters in test_iter:
            targets = encode_labels(disease_letters, label_map)

            images = images.to(device)
            vessel_masks = vessel_masks.to(device)
            inputs = build_model_input(images, vessel_masks, expected_in_channels)

            logits = model(inputs)
            preds = torch.argmax(logits, dim=1).detach().cpu()

            all_targets.extend(targets.tolist())
            all_preds.extend(preds.tolist())

    labels_sorted = sorted(inv_map.keys())

    accuracy_global = accuracy_score(all_targets, all_preds)
    precision_global, recall_global, f1_global, _ = precision_recall_fscore_support(
        all_targets,
        all_preds,
        labels=labels_sorted,
        average='weighted',
        zero_division=0,
    )

    precision_cls, recall_cls, f1_cls, support_cls = precision_recall_fscore_support(
        all_targets,
        all_preds,
        labels=labels_sorted,
        average=None,
        zero_division=0,
    )

    metrics_by_class = {}
    for i, cls_idx in enumerate(labels_sorted):
        cls_name = inv_map[cls_idx]
        metrics_by_class[cls_name] = {
            'precision': float(precision_cls[i]),
            'recall': float(recall_cls[i]),
            'f1': float(f1_cls[i]),
            'support': int(support_cls[i]),
        }

    metrics_global = {
        'accuracy': float(accuracy_global),
        'precision_weighted': float(precision_global),
        'recall_weighted': float(recall_global),
        'f1_weighted': float(f1_global),
    }

    return {
        'global': metrics_global,
        'by_class': metrics_by_class,
        'y_true': all_targets,
        'y_pred': all_preds,
    }

/home/msaunier/Documents/ITI5/MIA/MIA-eyes-vessel-segmentation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config img seule

In [6]:
# Chargement de la configuration
config_path = '../configs/config_img_classification.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration chargée :")
print(f"| Modèle chargé : {config['cnn']['name']}")
print(f"| Canaux d'entrée: {config['cnn']['in_channels']}")
print(f"| Canaux de sortie: {config['cnn']['out_channels']}")
print(f"| Epochs: {config['training']['n_epochs']}")
print(f"| Learning rate: {config['training']['learning_rate']}")
print(f"| Resume training: {config['training']['resume_training']}")

Configuration chargée :
| Modèle chargé : SimpleCNN
| Canaux d'entrée: 3
| Canaux de sortie: 4
| Epochs: 30
| Learning rate: 0.0001
| Resume training: False


In [7]:
# --------------------------
# Entraînement + sauvegarde (+ reprise possible)
# --------------------------
device = get_safe_device('cuda')
print(f"Device utilisé: {device}")
in_channels = config['cnn']['in_channels']
num_classes = config['cnn']['out_channels']

model = SimpleImageCNN(in_channels=in_channels, num_classes=num_classes)
optimizer = optim.Adam(model.parameters(), lr=config['training']['learning_rate'])

# Variables pour la reprise
initial_history = None
start_epoch = 0
model_dir = '../saved_models/classification_train_img'

if config['training'].get('resume_training', False):
    checkpoint_path = config['training'].get('resume_checkpoint_path')
    if checkpoint_path and os.path.isfile(checkpoint_path):
        try:
            model, optimizer, initial_history, start_epoch = load_classification_checkpoint(
                checkpoint_path=checkpoint_path,
                model=model,
                optimizer=optimizer,
                device=device,
            )
            model_dir = os.path.dirname(checkpoint_path)
            print("\n✓ Reprise activée avec succès")
            print(f"  Historique chargé: {len(initial_history['train_loss'])} epochs")
        except Exception as e:
            print(f"\n⚠ Erreur lors du chargement du checkpoint: {e}")
            print("  Démarrage d'un nouvel entraînement...")
    else:
        print("\n⚠ Chemin du checkpoint non spécifié ou fichier introuvable")
        print("  Démarrage d'un nouvel entraînement...")
else:
    print("\n  Démarrage d'un nouvel entraînement...")

trained_model, history, model_dir = train_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=config['training']['n_epochs'],
    learning_rate=config['training']['learning_rate'],
    device=device,
    save_root_dir=config['paths']['save_dir'],
    expected_in_channels=in_channels,
    num_classes=num_classes,
    model_dir=model_dir,
    optimizer=optimizer,
    initial_history=initial_history,
    start_epoch=start_epoch,
    )

print(f"Dossier de sauvegarde: {model_dir}")

Device utilisé: cuda

  Démarrage d'un nouvel entraînement...


Epoch 01/30 | train_loss=1.3444 train_acc=0.3583 | val_loss=1.1798 val_acc=0.5833


Epoch 02/30 | train_loss=1.0691 train_acc=0.5396 | val_loss=0.9386 val_acc=0.6167


Epoch 03/30 | train_loss=0.8686 train_acc=0.6521 | val_loss=0.8554 val_acc=0.6167


Epoch 04/30 | train_loss=0.7664 train_acc=0.7146 | val_loss=0.7752 val_acc=0.6333


...

Epoch 26/30 | train_loss=0.2420 train_acc=0.9125 | val_loss=0.5774 val_acc=0.8417


Epoch 27/30 | train_loss=0.2055 train_acc=0.9167 | val_loss=0.6518 val_acc=0.7333


Epoch 28/30 | train_loss=0.1890 train_acc=0.9208 | val_loss=0.7685 val_acc=0.7750


Epoch 29/30 | train_loss=0.1830 train_acc=0.9187 | val_loss=0.5946 val_acc=0.8167


Epoch 30/30 | train_loss=0.2032 train_acc=0.9187 | val_loss=0.7451 val_acc=0.7583
Meilleure val_acc: 0.8417 (epoch 26)
Best model: ../saved_models/classification_train_img/best_model.pth
Checkpoint: ../saved_models/classification_train_img/checkpoint.pth
Dossier de sauvegarde: ../saved_models/classification_train_img


In [8]:
# Chargement du meilleur modèle (optionnel, pour démontrer la réutilisation)
trained_model = SimpleImageCNN(in_channels=in_channels, num_classes=num_classes)

best_model_path = os.path.join(model_dir, 'best_model.pth')
if os.path.exists(best_model_path):
    print(f"Chargement du meilleur modèle depuis: {best_model_path}")
    best_state = torch.load(best_model_path, map_location=device)
    trained_model.load_state_dict(best_state)
else:
    print(f"Aucun meilleur modèle trouvé à: {best_model_path}. Utilisation du modèle entraîné actuel.")

# --------------------------
# Prédiction sur jeu de test + métriques détaillées
# --------------------------
test_results = predict_and_evaluate(
    model=trained_model,
    data_loader=test_loader,
    expected_in_channels=in_channels,
    device=device,
    label_map=label_to_idx,
    )

print("\n=== Métriques globales ===")
for metric_name, metric_value in test_results['global'].items():
    print(f"{metric_name}: {metric_value:.6f}")

print("\n=== Métriques par classe ===")
for cls_name, cls_metrics in test_results['by_class'].items():
    print(f"Classe {cls_name}:")
    print(f"  precision: {cls_metrics['precision']:.6f}")
    print(f"  recall:    {cls_metrics['recall']:.6f}")
    print(f"  f1:        {cls_metrics['f1']:.6f}")
    print(f"  support:   {cls_metrics['support']}")

Chargement du meilleur modèle depuis: ../saved_models/classification_train_img/best_model.pth



=== Métriques globales ===
accuracy: 0.750000
precision_weighted: 0.773133
recall_weighted: 0.750000
f1_weighted: 0.746211

=== Métriques par classe ===
Classe N:
  precision: 0.903226
  recall:    0.560000
  f1:        0.691358
  support:   50
Classe A:
  precision: 0.792453
  recall:    0.840000
  f1:        0.815534
  support:   50
Classe G:
  precision: 0.652174
  recall:    0.900000
  f1:        0.756303
  support:   50
Classe D:
  precision: 0.744681
  recall:    0.700000
  f1:        0.721649
  support:   50


## Image et mask

In [9]:
# Chargement de la configuration
config_path = '../configs/config_img_mask_classification.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration chargée :")
print(f"| Modèle chargé : {config['cnn']['name']}")
print(f"| Canaux d'entrée: {config['cnn']['in_channels']}")
print(f"| Canaux de sortie: {config['cnn']['out_channels']}")
print(f"| Epochs: {config['training']['n_epochs']}")
print(f"| Learning rate: {config['training']['learning_rate']}")
print(f"| Resume training: {config['training']['resume_training']}")

Configuration chargée :
| Modèle chargé : SimpleCNN
| Canaux d'entrée: 4
| Canaux de sortie: 4
| Epochs: 30
| Learning rate: 0.0001
| Resume training: False


In [10]:
# --------------------------
# Entraînement + sauvegarde (+ reprise possible)
# --------------------------
device = get_safe_device('cuda')
print(f"Device utilisé: {device}")
in_channels = config['cnn']['in_channels']
num_classes = config['cnn']['out_channels']

model = SimpleImageCNN(in_channels=in_channels, num_classes=num_classes)
optimizer = optim.Adam(model.parameters(), lr=config['training']['learning_rate'])

# Variables pour la reprise
initial_history = None
start_epoch = 0
model_dir = '../saved_models/classification_train_img_mask'

if config['training'].get('resume_training', False):
    checkpoint_path = config['training'].get('resume_checkpoint_path')
    if checkpoint_path and os.path.isfile(checkpoint_path):
        try:
            model, optimizer, initial_history, start_epoch = load_classification_checkpoint(
                checkpoint_path=checkpoint_path,
                model=model,
                optimizer=optimizer,
                device=device,
            )
            model_dir = os.path.dirname(checkpoint_path)
            print("\n✓ Reprise activée avec succès")
            print(f"  Historique chargé: {len(initial_history['train_loss'])} epochs")
        except Exception as e:
            print(f"\n⚠ Erreur lors du chargement du checkpoint: {e}")
            print("  Démarrage d'un nouvel entraînement...")
    else:
        print("\n⚠ Chemin du checkpoint non spécifié ou fichier introuvable")
        print("  Démarrage d'un nouvel entraînement...")
else:
    print("\n  Démarrage d'un nouvel entraînement...")

trained_model, history, model_dir = train_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=config['training']['n_epochs'],
    learning_rate=config['training']['learning_rate'],
    device=device,
    save_root_dir=config['paths']['save_dir'],
    expected_in_channels=in_channels,
    num_classes=num_classes,
    model_dir=model_dir,
    optimizer=optimizer,
    initial_history=initial_history,
    start_epoch=start_epoch,
    )

print(f"Dossier de sauvegarde: {model_dir}")

Device utilisé: cuda

  Démarrage d'un nouvel entraînement...


Epoch 01/30 | train_loss=1.3731 train_acc=0.3167 | val_loss=1.2136 val_acc=0.5333


Epoch 02/30 | train_loss=1.2613 train_acc=0.4083 | val_loss=1.2250 val_acc=0.4083


Epoch 03/30 | train_loss=1.1975 train_acc=0.4604 | val_loss=1.0792 val_acc=0.5583


Epoch 04/30 | train_loss=1.1306 train_acc=0.5042 | val_loss=1.0942 val_acc=0.5000


...

Epoch 26/30 | train_loss=0.4437 train_acc=0.8250 | val_loss=0.6967 val_acc=0.7250


Epoch 27/30 | train_loss=0.4176 train_acc=0.8438 | val_loss=0.6525 val_acc=0.7250


Epoch 28/30 | train_loss=0.3731 train_acc=0.8583 | val_loss=0.8409 val_acc=0.7167


Epoch 29/30 | train_loss=0.4164 train_acc=0.8250 | val_loss=0.6100 val_acc=0.7833


Epoch 30/30 | train_loss=0.3871 train_acc=0.8313 | val_loss=0.6568 val_acc=0.7583
Meilleure val_acc: 0.7833 (epoch 29)
Best model: ../saved_models/classification_train_img_mask/best_model.pth
Checkpoint: ../saved_models/classification_train_img_mask/checkpoint.pth
Dossier de sauvegarde: ../saved_models/classification_train_img_mask


In [11]:
# Chargement du meilleur modèle (optionnel, pour démontrer la réutilisation)
trained_model = SimpleImageCNN(in_channels=in_channels, num_classes=num_classes)

best_model_path = os.path.join(model_dir, 'best_model.pth')
if os.path.exists(best_model_path):
    print(f"Chargement du meilleur modèle depuis: {best_model_path}")
    best_state = torch.load(best_model_path, map_location=device)
    trained_model.load_state_dict(best_state)
else:
    print(f"Aucun meilleur modèle trouvé à: {best_model_path}. Utilisation du modèle entraîné actuel.")

# --------------------------
# Prédiction sur jeu de test + métriques détaillées
# --------------------------
test_results = predict_and_evaluate(
    model=trained_model,
    data_loader=test_loader,
    expected_in_channels=in_channels,
    device=device,
    label_map=label_to_idx,
    )

print("\n=== Métriques globales ===")
for metric_name, metric_value in test_results['global'].items():
    print(f"{metric_name}: {metric_value:.6f}")

print("\n=== Métriques par classe ===")
for cls_name, cls_metrics in test_results['by_class'].items():
    print(f"Classe {cls_name}:")
    print(f"  precision: {cls_metrics['precision']:.6f}")
    print(f"  recall:    {cls_metrics['recall']:.6f}")
    print(f"  f1:        {cls_metrics['f1']:.6f}")
    print(f"  support:   {cls_metrics['support']}")

Chargement du meilleur modèle depuis: ../saved_models/classification_train_img_mask/best_model.pth



=== Métriques globales ===
accuracy: 0.670000
precision_weighted: 0.683290
recall_weighted: 0.670000
f1_weighted: 0.640065

=== Métriques par classe ===
Classe N:
  precision: 0.705882
  recall:    0.240000
  f1:        0.358209
  support:   50
Classe A:
  precision: 0.803922
  recall:    0.820000
  f1:        0.811881
  support:   50
Classe G:
  precision: 0.630137
  recall:    0.920000
  f1:        0.747967
  support:   50
Classe D:
  precision: 0.593220
  recall:    0.700000
  f1:        0.642202
  support:   50


## Mask seul

In [12]:
# Chargement de la configuration
config_path = '../configs/config_mask_classification.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration chargée :")
print(f"| Modèle chargé : {config['cnn']['name']}")
print(f"| Canaux d'entrée: {config['cnn']['in_channels']}")
print(f"| Canaux de sortie: {config['cnn']['out_channels']}")
print(f"| Epochs: {config['training']['n_epochs']}")
print(f"| Learning rate: {config['training']['learning_rate']}")
print(f"| Resume training: {config['training']['resume_training']}")

Configuration chargée :
| Modèle chargé : SimpleCNN
| Canaux d'entrée: 1
| Canaux de sortie: 4
| Epochs: 30
| Learning rate: 0.0001
| Resume training: False


In [13]:
# --------------------------
# Entraînement + sauvegarde (+ reprise possible)
# --------------------------
device = get_safe_device('cuda')
print(f"Device utilisé: {device}")
in_channels = config['cnn']['in_channels']
num_classes = config['cnn']['out_channels']

model = SimpleImageCNN(in_channels=in_channels, num_classes=num_classes)
optimizer = optim.Adam(model.parameters(), lr=config['training']['learning_rate'])

# Variables pour la reprise
initial_history = None
start_epoch = 0
model_dir = '../saved_models/classification_train_mask'

if config['training'].get('resume_training', False):
    checkpoint_path = config['training'].get('resume_checkpoint_path')
    if checkpoint_path and os.path.isfile(checkpoint_path):
        try:
            model, optimizer, initial_history, start_epoch = load_classification_checkpoint(
                checkpoint_path=checkpoint_path,
                model=model,
                optimizer=optimizer,
                device=device,
            )
            model_dir = os.path.dirname(checkpoint_path)
            print("\n✓ Reprise activée avec succès")
            print(f"  Historique chargé: {len(initial_history['train_loss'])} epochs")
        except Exception as e:
            print(f"\n⚠ Erreur lors du chargement du checkpoint: {e}")
            print("  Démarrage d'un nouvel entraînement...")
    else:
        print("\n⚠ Chemin du checkpoint non spécifié ou fichier introuvable")
        print("  Démarrage d'un nouvel entraînement...")
else:
    print("\n  Démarrage d'un nouvel entraînement...")

trained_model, history, model_dir = train_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=config['training']['n_epochs'],
    learning_rate=config['training']['learning_rate'],
    device=device,
    save_root_dir=config['paths']['save_dir'],
    expected_in_channels=in_channels,
    num_classes=num_classes,
    model_dir=model_dir,
    optimizer=optimizer,
    initial_history=initial_history,
    start_epoch=start_epoch,
    )

print(f"Dossier de sauvegarde: {model_dir}")

Device utilisé: cuda

  Démarrage d'un nouvel entraînement...


Epoch 01/30 | train_loss=1.4357 train_acc=0.2667 | val_loss=1.4130 val_acc=0.2250


Epoch 02/30 | train_loss=1.3983 train_acc=0.2604 | val_loss=1.4726 val_acc=0.2000


Epoch 03/30 | train_loss=1.3768 train_acc=0.3146 | val_loss=1.3748 val_acc=0.2833


Epoch 04/30 | train_loss=1.3663 train_acc=0.3125 | val_loss=1.4320 val_acc=0.2083


...

Epoch 26/30 | train_loss=1.0459 train_acc=0.5125 | val_loss=1.1845 val_acc=0.4083


Epoch 27/30 | train_loss=1.0262 train_acc=0.5208 | val_loss=1.0757 val_acc=0.4750


Epoch 28/30 | train_loss=1.0374 train_acc=0.4833 | val_loss=1.0870 val_acc=0.4417


Epoch 29/30 | train_loss=1.0345 train_acc=0.5271 | val_loss=1.1097 val_acc=0.3750


Epoch 30/30 | train_loss=1.0166 train_acc=0.5083 | val_loss=1.0762 val_acc=0.5167
Meilleure val_acc: 0.5167 (epoch 20)
Best model: ../saved_models/classification_train_mask/best_model.pth
Checkpoint: ../saved_models/classification_train_mask/checkpoint.pth
Dossier de sauvegarde: ../saved_models/classification_train_mask


In [14]:
# Chargement du meilleur modèle (optionnel, pour démontrer la réutilisation)
trained_model = SimpleImageCNN(in_channels=in_channels, num_classes=num_classes)

best_model_path = os.path.join(model_dir, 'best_model.pth')
if os.path.exists(best_model_path):
    print(f"Chargement du meilleur modèle depuis: {best_model_path}")
    best_state = torch.load(best_model_path, map_location=device)
    trained_model.load_state_dict(best_state)
else:
    print(f"Aucun meilleur modèle trouvé à: {best_model_path}. Utilisation du modèle entraîné actuel.")

# --------------------------
# Prédiction sur jeu de test + métriques détaillées
# --------------------------
test_results = predict_and_evaluate(
    model=trained_model,
    data_loader=test_loader,
    expected_in_channels=in_channels,
    device=device,
    label_map=label_to_idx,
    )

print("\n=== Métriques globales ===")
for metric_name, metric_value in test_results['global'].items():
    print(f"{metric_name}: {metric_value:.6f}")

print("\n=== Métriques par classe ===")
for cls_name, cls_metrics in test_results['by_class'].items():
    print(f"Classe {cls_name}:")
    print(f"  precision: {cls_metrics['precision']:.6f}")
    print(f"  recall:    {cls_metrics['recall']:.6f}")
    print(f"  f1:        {cls_metrics['f1']:.6f}")
    print(f"  support:   {cls_metrics['support']}")

Chargement du meilleur modèle depuis: ../saved_models/classification_train_mask/best_model.pth



=== Métriques globales ===
accuracy: 0.475000
precision_weighted: 0.524403
recall_weighted: 0.475000
f1_weighted: 0.477165

=== Métriques par classe ===
Classe N:
  precision: 0.568627
  recall:    0.580000
  f1:        0.574257
  support:   50
Classe A:
  precision: 0.333333
  recall:    0.520000
  f1:        0.406250
  support:   50
Classe G:
  precision: 0.500000
  recall:    0.480000
  f1:        0.489796
  support:   50
Classe D:
  precision: 0.695652
  recall:    0.320000
  f1:        0.438356
  support:   50
